# 📊 CMF Financial JSON → XLSX Converter

Converts extracted CMF financial statement JSON files into formatted Excel workbooks.

Each JSON produces one `.xlsx` with one sheet per financial statement:

| Sheet | JSON key |
|---|---|
| **Bilan Actif** | `bilan_actif` |
| **Bilan Passif** | `bilan_passif` |
| **Etat de Résultat** | `cpc` or `etat_resultat` |
| **Flux de Trésorerie** | `tft` or `flux_tresorerie` |

---

## 1. Install Dependencies

In [1]:
!pip install openpyxl -q
print("✅ Dependencies ready.")

✅ Dependencies ready.


## 2. Imports

In [2]:
import json
import re
import os
import glob
from pathlib import Path

from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

print("✅ Imports done.")

✅ Imports done.


## 3. Configuration

> **Set your input and output folder paths here.**

In [9]:
# ─── USER CONFIGURATION ───────────────────────────────────────────────────────

INPUT_FOLDER  = r"CMF_SITEX_PDFs"          # ← folder with JSON files
OUTPUT_FOLDER = r"CMF_SITEX_PDFs" # ← where XLSX files go

# ──────────────────────────────────────────────────────────────────────────────

os.makedirs(OUTPUT_FOLDER, exist_ok=True)
print(f"Input  folder : {INPUT_FOLDER}")
print(f"Output folder : {OUTPUT_FOLDER}")

Input  folder : CMF_SITEX_PDFs
Output folder : CMF_SITEX_PDFs


## 4. Style Constants & Helpers

In [10]:
# ── Color palette ─────────────────────────────────────────────────────────────
C_DARK_BLUE  = "1F4E79"
C_MID_BLUE   = "2E75B6"
C_LIGHT_BLUE = "D6E4F0"
C_TOTAL_BG   = "BDD7EE"
C_WHITE      = "FFFFFF"
C_BLACK      = "000000"

THIN = Side(style="thin",   color="B8CCE4")
THIK = Side(style="medium", color=C_DARK_BLUE)

# ── Sheet name mapping ────────────────────────────────────────────────────────
SHEET_NAMES = {
    "bilan_actif":     "Bilan Actif",
    "bilan_passif":    "Bilan Passif",
    "cpc":             "Etat de Résultat",
    "tft":             "Flux de Trésorerie",
    "etat_resultat":   "Etat de Résultat",
    "flux_tresorerie": "Flux de Trésorerie",
}

def _border():
    return Border(left=THIN, right=THIN, top=THIN, bottom=THIN)

def _fill(color):
    return PatternFill("solid", start_color=color, end_color=color)

def _font(bold=False, color=C_BLACK, size=10):
    return Font(bold=bold, color=color, size=size, name="Arial")

print("✅ Style constants and helpers defined.")

✅ Style constants and helpers defined.


## 5. Core Logic Functions

In [11]:
def extract_year(filename: str) -> str:
    """Extract the 4-digit year from a filename."""
    m = re.search(r'(20[1-9]\d)', filename)
    if m:
        return m.group(1)
    m = re.search(r'\d{4}(\d{2})(?=_|\.)', filename)
    if m:
        return str(2000 + int(m.group(1)))
    return "unknown"


def resolve_val_keys(raw_headers: list, sample_values: dict) -> list:
    """
    Map each raw header string to the actual key used in the values dict.
    Headers may say 'Exercice clos le 31décembre 2015' but the
    values dict uses the short key '2015'. We match by substring.
    """
    actual_keys = list(sample_values.keys())
    resolved = []
    for h in raw_headers:
        h_clean = h.replace("\n", " ").strip()
        if h_clean in actual_keys:
            resolved.append(h_clean)
            continue
        match = next((k for k in actual_keys if k in h_clean or k in h), None)
        resolved.append(match if match else h)
    return resolved


def section_to_rows(section: dict):
    """
    Parse a financial_statements section into (headers, rows).
    Handles two table styles:
      - headers = ['Notes', '31/12/XXXX', ...] with label as a separate key
      - headers = ['label', 'Notes', '31/12/XXXX', ...] with label inside headers
    """
    all_headers = []
    all_rows    = []

    for table in section.get("tables", []):
        caption = (table.get("caption") or "").strip()
        if caption:
            all_rows.append({"label": caption, "values": {},
                             "val_keys": [], "is_section": True, "is_total": False})

        raw_headers = table.get("headers") or []
        if raw_headers and raw_headers[0].lower() == "label":
            raw_headers = raw_headers[1:]

        sample_values = {}
        for row in table.get("rows", []):
            v = row.get("values") or {}
            if v:
                sample_values = v
                break

        val_keys    = resolve_val_keys(raw_headers, sample_values)
        col_headers = ["Libellé"] + [h.replace("\n", " ").strip() for h in raw_headers]

        if not all_headers and col_headers:
            all_headers = col_headers

        for row in table.get("rows", []):
            label  = (row.get("label") or "").strip()
            values = row.get("values") or {}
            is_total   = label.upper().startswith("TOTAL")
            is_section = bool(
                label and (not values or all(v in ("", None) for v in values.values()))
            )
            all_rows.append({
                "label":      label,
                "values":     values,
                "val_keys":   val_keys,
                "is_section": is_section and not is_total,
                "is_total":   is_total,
            })

    return all_headers, all_rows


print("✅ Core logic functions defined.")

✅ Core logic functions defined.


## 6. Sheet Writer

In [12]:
def write_sheet(ws, title: str, headers: list, rows: list):
    """Write a financial statement section to an openpyxl worksheet."""
    num_cols = len(headers) if headers else 4

    # ── Title row ─────────────────────────────────────────────────────────────
    ws.append([title])
    r = ws.max_row
    if num_cols > 1:
        ws.merge_cells(start_row=r, start_column=1, end_row=r, end_column=num_cols)
    c = ws.cell(row=r, column=1)
    c.font      = _font(bold=True, color=C_WHITE, size=13)
    c.fill      = _fill(C_DARK_BLUE)
    c.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    ws.row_dimensions[r].height = 34

    if not headers:
        ws.append(["Aucune donnée disponible"])
        return

    # ── Column headers ────────────────────────────────────────────────────────
    ws.append(headers)
    r = ws.max_row
    for i, h in enumerate(headers, start=1):
        c = ws.cell(row=r, column=i)
        c.font      = _font(bold=True, color=C_WHITE)
        c.fill      = _fill(C_MID_BLUE)
        c.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        c.border    = _border()
    ws.row_dimensions[r].height = 22

    # ── Data rows ─────────────────────────────────────────────────────────────
    alt = False
    for row in rows:
        label      = row["label"]
        values     = row["values"]
        val_keys   = row.get("val_keys") or headers[1:]
        is_section = row["is_section"]
        is_total   = row["is_total"]

        if is_section:
            ws.append([label])
            r = ws.max_row
            ws.merge_cells(start_row=r, start_column=1, end_row=r, end_column=num_cols)
            c = ws.cell(row=r, column=1)
            c.font      = _font(bold=True, color=C_WHITE)
            c.fill      = _fill(C_MID_BLUE)
            c.alignment = Alignment(horizontal="left", vertical="center")
            c.border    = Border(top=THIK, bottom=THIK, left=THIN, right=THIN)
            ws.row_dimensions[r].height = 18
            alt = False
            continue

        row_data = [label] + [values.get(k, "") for k in val_keys]
        ws.append(row_data)
        r  = ws.max_row
        bg = C_TOTAL_BG if is_total else (C_LIGHT_BLUE if alt else C_WHITE)

        for i, _ in enumerate(row_data, start=1):
            c = ws.cell(row=r, column=i)
            c.fill   = _fill(bg)
            c.border = _border()
            c.font   = _font(bold=is_total)
            if i == 1:
                c.alignment = Alignment(horizontal="left", vertical="center",
                                        wrap_text=True, indent=1)
            else:
                c.alignment = Alignment(horizontal="right", vertical="center")
        ws.row_dimensions[r].height = 16
        if not is_total:
            alt = not alt

    # ── Auto column widths ────────────────────────────────────────────────────
    for col in ws.columns:
        max_len    = 0
        col_letter = get_column_letter(col[0].column)
        for cell in col:
            try:
                if cell.value:
                    max_len = max(max_len, len(str(cell.value)))
            except Exception:
                pass
        if col[0].column == 1:
            ws.column_dimensions[col_letter].width = min(max_len + 4, 55)
        else:
            ws.column_dimensions[col_letter].width = max(max_len + 4, 18)

    ws.freeze_panes = "A3"


print("✅ write_sheet() defined.")

✅ write_sheet() defined.


## 7. File Processor

In [13]:
def process_file(json_path: str, output_dir: str):
    """
    Convert one JSON file to XLSX.
    Returns the output path on success, None if skipped or empty.
    Supports two JSON formats:
      Format A: {"financial_statements": {"bilan_actif": {...}, ...}}
      Format B: {"pages": [{"page_type": "bilan_actif", ...}, ...]}
    """
    filename = Path(json_path).stem
    year     = extract_year(filename)
    out_name = f"CMF_NewBodyLine_{year}.xlsx"
    out_path = os.path.join(output_dir, out_name)

    if os.path.exists(out_path):
        print(f"  ⏭️  [{year}]  {out_name} already exists — skipped.")
        return None

    print(f"  [{year}]  {filename}  →  {out_name}")

    with open(json_path, encoding="utf-8") as f:
        data = json.load(f)

    if "financial_statements" in data:
        financial = data["financial_statements"]
    elif "pages" in data:
        financial = {p["page_type"]: p for p in data["pages"] if p.get("page_type")}
    else:
        financial = {}

    if not financial:
        print(f"         ⚠  No 'financial_statements' or 'pages' key — skipping.")
        return None

    wb = Workbook()
    wb.remove(wb.active)   # remove default blank sheet

    for key, section in financial.items():
        sheet_name = SHEET_NAMES.get(key, key[:31])
        ws         = wb.create_sheet(title=sheet_name)
        title      = (section.get("title") or sheet_name).replace("\n", " — ")
        headers, rows = section_to_rows(section)
        write_sheet(ws, title, headers, rows)

    if not wb.sheetnames:
        print(f"         ⚠  No sheets created — skipping.")
        return None

    wb.save(out_path)
    return out_path


print("✅ process_file() defined.")

✅ process_file() defined.


## 8. Preview — JSON Files Found

Lists all JSON files that will be processed before running the conversion.

In [14]:
import pandas as pd

json_files = sorted(glob.glob(os.path.join(INPUT_FOLDER, "*.json")))

if not json_files:
    print(f"❌ No JSON files found in:\n   {INPUT_FOLDER}")
else:
    rows = []
    for jf in json_files:
        fname = Path(jf).name
        year  = extract_year(Path(jf).stem)
        size_kb = os.path.getsize(jf) // 1024
        out_name = f"CMF_NewBodyLine_{year}.xlsx"
        already  = "✅ exists" if os.path.exists(os.path.join(OUTPUT_FOLDER, out_name)) else "🔄 to convert"
        rows.append({"File": fname, "Year": year, "Size (KB)": size_kb, "Status": already})

    preview_df = pd.DataFrame(rows)
    print(f"📂 Found {len(json_files)} JSON file(s):\n")
    display(preview_df)

📂 Found 10 JSON file(s):



,File,Year,Size (KB),Status
0,sitex-efd-3112-2020_extract_pages2-5.json,2020,32,🔄 to convert
1,sitex_efd311215_extract_pages2-5.json,2015,32,🔄 to convert
2,sitex_efd311216_extract_pages2-5.json,2016,32,🔄 to convert
3,sitex_efd311217_extract_pages2-5.json,2017,32,🔄 to convert
4,sitex_efd311218_extract_pages2-5.json,2018,30,🔄 to convert
5,sitex_efd311219_extract_pages2-5.json,2019,36,🔄 to convert
6,sitex_efd311221_extract_pages2-5.json,2021,31,🔄 to convert
7,sitex_efd311222_extract_pages2-5.json,2022,33,🔄 to convert
8,sitex_efd311223_extract_pages2-5.json,2023,32,🔄 to convert
9,sitex_efd311224_extract_pages2-5.json,2024,26,🔄 to convert


## 9. Run — Convert All JSON → XLSX

Files that already have an XLSX output are **skipped automatically**.

In [15]:
if not json_files:
    print("No files to process. Check INPUT_FOLDER in Cell 3.")
else:
    print(f"Converting {len(json_files)} JSON file(s)...\n")
    created, skipped, errors = [], [], []

    for jf in json_files:
        try:
            out = process_file(jf, OUTPUT_FOLDER)
            if out:
                created.append(out)
            else:
                skipped.append(Path(jf).name)
        except Exception as e:
            name = Path(jf).name
            print(f"  ❌ ERROR  {name}: {e}")
            errors.append((name, str(e)))

    # ── Summary ───────────────────────────────────────────────────────────────
    print(f"\n{'=' * 55}")
    print(f"✅  {len(created)} XLSX file(s) created in:")
    print(f"   {OUTPUT_FOLDER}")

    if skipped:
        print(f"\n⏭️   {len(skipped)} file(s) skipped (already exist):")
        for s in skipped:
            print(f"   {s}")

    if errors:
        print(f"\n⚠️  {len(errors)} file(s) failed:")
        for fname, err in errors:
            print(f"   {fname}: {err}")

Converting 10 JSON file(s)...

  [2020]  sitex-efd-3112-2020_extract_pages2-5  →  CMF_NewBodyLine_2020.xlsx
  [2015]  sitex_efd311215_extract_pages2-5  →  CMF_NewBodyLine_2015.xlsx
  [2016]  sitex_efd311216_extract_pages2-5  →  CMF_NewBodyLine_2016.xlsx
  [2017]  sitex_efd311217_extract_pages2-5  →  CMF_NewBodyLine_2017.xlsx
  [2018]  sitex_efd311218_extract_pages2-5  →  CMF_NewBodyLine_2018.xlsx
  [2019]  sitex_efd311219_extract_pages2-5  →  CMF_NewBodyLine_2019.xlsx
  [2021]  sitex_efd311221_extract_pages2-5  →  CMF_NewBodyLine_2021.xlsx
  [2022]  sitex_efd311222_extract_pages2-5  →  CMF_NewBodyLine_2022.xlsx
  [2023]  sitex_efd311223_extract_pages2-5  →  CMF_NewBodyLine_2023.xlsx
  [2024]  sitex_efd311224_extract_pages2-5  →  CMF_NewBodyLine_2024.xlsx

✅  10 XLSX file(s) created in:
   CMF_SITEX_PDFs


---
## ✅ Done!

Each JSON is now an `.xlsx` file with colour-coded sheets:

| Row type | Style |
|---|---|
| **Title** | Dark blue background, white bold text |
| **Column headers** | Mid blue background, white bold text |
| **Section headers** | Mid blue background, white bold text (full width) |
| **TOTAL rows** | Light blue background, bold text |
| **Data rows** | Alternating white / light blue |

**Tips:**
- Add more keys to `SHEET_NAMES` in Cell 4 to handle custom section names from your JSONs.
- Use Cell 11 to re-convert a single file without reprocessing the whole batch.
- The converter handles both `financial_statements` and `pages` JSON formats automatically.